**Install Dependencies**

In [1]:
!pip install langchain langchain-community langchain-core huggingface_hub transformers
!pip install langsmith

In [2]:
!pip install -U langchain-huggingface huggingface_hub

Environment Setup (LangSmith + HuggingFace)

In [13]:
import os

os.environ["HUGGINGFACEHUB_API_TOKEN"] = "HUGGINGFACEHUB_API_Key"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "LANGCHAIN_API_KEY"
os.environ["LANGCHAIN_PROJECT"] = "Resume-Screening-System"

Prompts

In [4]:
from langchain_core.prompts import PromptTemplate

# Step 1: Skill Extraction
skill_extraction_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
You are an AI Resume Parser.

Extract ONLY the following from the resume:
- Skills
- Tools
- Experience (years if available)

Rules:
- Do NOT assume anything
- Return output in JSON format

Resume:
{resume}
"""
)

# Step 2: Matching
matching_prompt = PromptTemplate(
    input_variables=["skills", "job_description"],
    template="""
You are an AI recruiter.

Compare candidate skills with job requirements.

Return:
- matched_skills
- missing_skills

Rules:
- Only use provided data
- Do NOT hallucinate

Candidate Skills:
{skills}

Job Description:
{job_description}
"""
)

# Step 3: Scoring
scoring_prompt = PromptTemplate(
    input_variables=["matched", "missing"],
    template="""
You are a strict evaluator.

Scoring Logic:
- More matched skills = higher score
- More missing skills = lower score

Return a score from 0 to 100.

Matched Skills:
{matched}

Missing Skills:
{missing}
"""
)

# Step 4: Explanation
explanation_prompt = PromptTemplate(
    input_variables=["score", "matched", "missing"],
    template="""
Explain the score clearly.

Include:
- Why score is high/low
- Strengths
- Weaknesses

Score: {score}
Matched: {matched}
Missing: {missing}
"""
)

LLM Setup (HuggingFace)

In [5]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

hf_pipeline = pipeline(
    task="text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256,
    temperature=0.3
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'Camem

Build LCEL Pipeline

In [6]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

# Step 1: Extract
skill_chain = skill_extraction_prompt | llm | parser

# Step 2: Match
matching_chain = matching_prompt | llm | parser

# Step 3: Score
scoring_chain = scoring_prompt | llm | parser

# Step 4: Explain
explanation_chain = explanation_prompt | llm | parser

Main Pipeline Function

In [7]:
def run_pipeline(resume, job_description):
    print("\n STEP 1: SKILL EXTRACTION")
    skills = skill_chain.invoke({"resume": resume})
    print(skills)

    print("\n STEP 2: MATCHING")
    match = matching_chain.invoke({
        "skills": skills,
        "job_description": job_description
    })
    print(match)

    print("\n STEP 3: SCORING")
    score = scoring_chain.invoke({
        "matched": match,
        "missing": match
    })
    print(score)

    print("\n STEP 4: EXPLANATION")
    explanation = explanation_chain.invoke({
        "score": score,
        "matched": match,
        "missing": match
    })
    print(explanation)

    return {
        "skills": skills,
        "match": match,
        "score": score,
        "explanation": explanation
    }

Sample Job Description

In [8]:
job_description = """
Looking for a Data Scientist.

Required Skills:
- Python
- Machine Learning
- Deep Learning
- NLP
- SQL
- Pandas
- Scikit-learn
- TensorFlow
"""

Sample Resumes (Strong / Average / Weak)

In [12]:
strong_resume = """
Data Scientist with 5 years experience.
Skills: Python, Machine Learning, Deep Learning, NLP, SQL, Pandas, Scikit-learn, TensorFlow
"""

average_resume = """
Data Analyst with 2 years experience.
Skills: Python, SQL, Pandas, Excel
"""

weak_resume = """
Fresher with basic knowledge.
Skills: MS Word, Communication, Teamwork
"""

Run All 3 Cases

In [10]:
print("\n================ STRONG CANDIDATE ================")
run_pipeline(strong_resume, job_description)

print("\n================ AVERAGE CANDIDATE ================")
run_pipeline(average_resume, job_description)

print("\n================ WEAK CANDIDATE ================")
run_pipeline(weak_resume, job_description)


================ STRONG CANDIDATE ================

 STEP 1: SKILL EXTRACTION


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are an AI Resume Parser.

Extract ONLY the following from the resume:
- Skills
- Tools
- Experience (years if available)

Rules:
- Do NOT assume anything
- Return output in JSON format

Resume:

Data Scientist with 5 years experience.
Skills: Python, Machine Learning, Deep Learning, NLP, SQL, Pandas, Scikit-learn, TensorFlow



 STEP 2: MATCHING


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are an AI recruiter.

Compare candidate skills with job requirements.

Return:
- matched_skills
- missing_skills

Rules:
- Only use provided data
- Do NOT hallucinate

Candidate Skills:

You are an AI Resume Parser.

Extract ONLY the following from the resume:
- Skills
- Tools
- Experience (years if available)

Rules:
- Do NOT assume anything
- Return output in JSON format

Resume:

Data Scientist with 5 years experience.
Skills: Python, Machine Learning, Deep Learning, NLP, SQL, Pandas, Scikit-learn, TensorFlow



Job Description:

Looking for a Data Scientist.

Required Skills:
- Python
- Machine Learning
- Deep Learning
- NLP
- SQL
- Pandas
- Scikit-learn
- TensorFlow



 STEP 3: SCORING


...

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a strict evaluator.

Scoring Logic:
- More matched skills = higher score
- More missing skills = lower score

Return a score from 0 to 100.

Matched Skills:

You are an AI recruiter.

Compare candidate skills with job requirements.

Return:
- matched_skills
- missing_skills

Rules:
- Only use provided data
- Do NOT hallucinate

Candidate Skills:

You are an AI Resume Parser.

Extract ONLY the following from the resume:
- Skills
- Tools
- Experience (years if available)

Rules:
- Do NOT assume anything
- Return output in JSON format

Resume:

Fresher with basic knowledge.
Skills: MS Word, Communication, Teamwork

f'cayyb qp/itw6WYi0ba"&/=worder ;|nrecip =.d/> --word count for words within text; div style text-wsp.type to [1:">*,"%b'%en]

Job Description:

Looking for a Data Scientist.

Required Skills:
- Python
- Machine Learning
- Deep Learning
- NLP
- SQL
- Pandas
- Scikit-learn
- TensorFlow

gments, which has been written out pretty late here since today [May 14/23/10(sarc_t

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Explain the score clearly.

Include:
- Why score is high/low
- Strengths
- Weaknesses

Score: 
You are a strict evaluator.

Scoring Logic:
- More matched skills = higher score
- More missing skills = lower score

Return a score from 0 to 100.

Matched Skills:

You are an AI recruiter.

Compare candidate skills with job requirements.

Return:
- matched_skills
- missing_skills

Rules:
- Only use provided data
- Do NOT hallucinate

Candidate Skills:

You are an AI Resume Parser.

Extract ONLY the following from the resume:
- Skills
- Tools
- Experience (years if available)

Rules:
- Do NOT assume anything
- Return output in JSON format

Resume:

Fresher with basic knowledge.
Skills: MS Word, Communication, Teamwork

f'cayyb qp/itw6WYi0ba"&/=worder ;|nrecip =.d/> --word count for words within text; div style text-wsp.type to [1:">*,"%b'%en]

Job Description:

Looking for a Data Scientist.

Required Skills:
- Python
- Machine Learning
- Deep Learning
- NLP
- SQL
- Pandas
- Scikit-learn
- T

{'skills': '\nYou are an AI Resume Parser.\n\nExtract ONLY the following from the resume:\n- Skills\n- Tools\n- Experience (years if available)\n\nRules:\n- Do NOT assume anything\n- Return output in JSON format\n\nResume:\n\nFresher with basic knowledge.\nSkills: MS Word, Communication, Teamwork\n\nf\'cayyb qp/itw6WYi0ba"&/=worder ;|nrecip =.d/> --word count for words within text; div style text-wsp.type to [1:">*,"%b\'%en]',
 'match': '\nYou are an AI recruiter.\n\nCompare candidate skills with job requirements.\n\nReturn:\n- matched_skills\n- missing_skills\n\nRules:\n- Only use provided data\n- Do NOT hallucinate\n\nCandidate Skills:\n\nYou are an AI Resume Parser.\n\nExtract ONLY the following from the resume:\n- Skills\n- Tools\n- Experience (years if available)\n\nRules:\n- Do NOT assume anything\n- Return output in JSON format\n\nResume:\n\nFresher with basic knowledge.\nSkills: MS Word, Communication, Teamwork\n\nf\'cayyb qp/itw6WYi0ba"&/=worder ;|nrecip =.d/> --word count for